# OpenClaw в Guildclaw — короткий гайд

**OpenClaw** здесь — шлюз + агент поверх **локальной модели** (llama-server, GGUF). Этот ноутбук — шпаргалка по настройке и ссылкам.

**Где что лежит (в контейнере):**

| Что | Путь |
|-----|------|
| Конфиг OpenClaw | `/workspace/.openclaw/openclaw.json` (часто тот же каталог, что `~/.openclaw`) |
| Рабочая зона агента (файлы, AGENTS.md) | `/workspace/openclaw` |
| Активная GGUF-модель (Model Hub) | `/workspace/.guildclaw/active.json` |
| Файлы моделей | `/workspace/models/gguf/` |

**Порты:**

| Порт | Сервис |
|------|--------|
| **8080** | Model Hub — скачать/активировать GGUF |
| **8000** | LLM API (OpenAI-совместимый) |
| **18789** | OpenClaw Web UI (Control UI) + WebSocket шлюза |
| **8081** | Node pairing (только очередь **нод**, не device) |
| **8888** | JupyterLab (этот интерфейс) |

На **RunPod** вместо `localhost` — `https://<POD_ID>-<порт>.proxy.runpod.net`.

## 1. Первый запуск: модель

1. Открой **Model Hub** (порт **8080**). Если задан `MODEL_HUB_TOKEN` — добавь `?token=…` в URL или введи токен на странице входа.
2. Скачай пресет или укажи URL на `.gguf`, нажми **Активировать**.
3. Подожди перезапуск **llama-server** — в `openclaw.json` провайдер `local-llama` подхватит модель (синхронизация при старте пода / скриптом).

Контекст и компакция задаются env: **`LLAMA_CTX_SIZE`**, **`OPENCLAW_COMPACTION_*`** — см. **DEPLOY.md** в репозитории Guildclaw.

## 2. Веб-интерфейс и токен

- **OpenClaw UI:** `http://localhost:18789/?token=<OPENCLAW_WEB_PASSWORD>`
- Тот же пароль часто используется как токен шлюза для клиентов.

**Device pairing** (браузер, CLI): одобрение в UI :18789 или `openclaw devices pending` → `openclaw devices approve <id>`.

**Node pairing** (мобильная нода по WS): отдельная очередь; страница **:8081** показывает только то, что пришло через **`node.pair.request`** на шлюз **:18789** (`wss://…`). С телеграмом и Control UI это **не** смешивается.

## 3. Telegram-бот

- Задай **`TELEGRAM_BOT_TOKEN`** в переменных окружения пода (токен от @BotFather).
- **Перезапусти под** — в актуальном образе токен дописывается в `channels.telegram` в `openclaw.json`.
- Либо правь `openclaw.json` вручную: `channels.telegram.botToken` и `enabled: true`.

Бот — это **канал**, не **node pairing** на :8081.

## 4. Правка конфига и помощник

- Редактор в образе: **`nano`** (или `vi`). Пример: `nano /workspace/.openclaw/openclaw.json`
- В терминале пода: **`guildclaw-mate`** — ссылки, `doctor`, `telegram`, проброс **`guildclaw-mate oc …`** → `openclaw`.
- Полный путь к скрипту: `python3 /opt/guildclaw/scripts/guildclaw_mate.py info`

Ниже — опционально: выполнить сводку **Mate** из ячейки.

In [ ]:
import shutil
import subprocess

exe = shutil.which("guildclaw-mate") or "/workspace/guildclaw-mate"
subprocess.run([exe, "info"], cwd="/workspace")

## 5. Полезное

- Документация образа и env: **`DEPLOY.md`** (репозиторий Guildclaw).
- Node pairing (протокол): [Gateway-owned pairing](https://openclaws.io/docs/gateway/pairing/)
- В этой папке также **`Guildclaw_Mate.ipynb`** — ячейки с проверками сервисов.
- Правила агента для чата: **`AGENTS.md`**, **`IDENTITY.md`**.